# Run omnipose segmentation on WEXAC

Submits one LSF (`bsub`) job per phase image. Each job runs `lsf_omnipose_seg_morph.py` on a single FOV and saves:
- `{fov_name}.seg.npy` — labeled mask (working copy)
- `{fov_name}.seg.unfilt.npy` — raw model output
- `{fov_name}.props.txt` — morphological properties per cell
- `{fov_name}.phase_per_cell.txt` — phase intensity stats per cell
- `{fov_name}.seg_image.tif` — input image saved alongside the mask

---

## How to run this notebook on WEXAC

**Option A — WEXAC JupyterHub (recommended, no SSH needed)**
1. Open a browser and go to: `https://jupyter.wexac.weizmann.ac.il`
2. Log in with your Weizmann account.
3. Navigate to this notebook file and open it.
4. Select the `omnipose` kernel (top-right menu → *Change Kernel*).
5. Run cells top to bottom.

**Option B — SSH from terminal**
1. Open a terminal and connect: `ssh <username>@wexac.weizmann.ac.il`
2. Activate the environment: `conda activate omnipose`
3. Navigate to the notebook folder and start Jupyter: `jupyter notebook --no-browser --port=8888`
4. Follow the tunnel/URL instructions printed in the terminal.

> **Note:** This notebook submits jobs — it does NOT run segmentation itself. You can close it after submitting.

## Settings

| Variable | What it controls |
|---|---|
| `PHASE_DIR` | Folder on WEXAC containing `.phase.tif` files |
| `OUTPUT_DIR` | Folder where `omnipose_seg/fov_*_hyb_*` subfolders will be created |
| `LSF_SCRIPT` | Path to the worker script on WEXAC |
| `PATTERN` | Glob to select which TIFs to process (e.g. `"*.phase.tif"` for all, `"fov_1_hyb_1.phase.tif"` for one) |
| `MODEL_TYPE` | Omnipose model name |
| `IS_OVERWRITE` | `False` = skip FOVs already segmented; `True` = re-run everything |
| `LSF_QUEUE` | WEXAC queue — `"short"` for jobs under 1 hour, `"long"` for longer ones |
| `LSF_MEMORY` | RAM per job in MB — 16 GB is enough for a 2048×2048 phase image |
| `CONDA_ENV` | Conda environment on WEXAC that has omnipose installed |

In [ ]:
from pathlib import Path

# ── WEXAC paths ───────────────────────────────────────────────────────────────
_base       = Path("/home/labs/danielda/danielda/analysis/zp_ecoli_M902_p2f_020625")
PHASE_DIR   = _base / "phase-proj"          # folder with .phase.tif files
OUTPUT_DIR  = _base / "omnipose_seg"         # where per-FOV subfolders will be saved
LSF_SCRIPT  = Path("/home/labs/danielda/danielda/scripts/pipeline/lsf_functions/lsf_omnipose_seg_morph.py")

# ── which images to process ───────────────────────────────────────────────────
PATTERN     = "*.phase.tif"   # "*.phase.tif" = all FOVs  |  "fov_1_hyb_1.phase.tif" = one FOV

# ── omnipose settings ─────────────────────────────────────────────────────────
MODEL_TYPE  = "bact_phase_omni"
IS_OVERWRITE = False   # False = skip already-segmented FOVs

# ── LSF job settings ──────────────────────────────────────────────────────────
LSF_QUEUE   = "short"    # short = up to 1 h wall time
LSF_MEMORY  = 16000      # MB per job
CONDA_ENV   = "omnipose"

## Find phase images

Lists all `.phase.tif` files matching `PATTERN` in `PHASE_DIR`.
Check the count before submitting — if it looks wrong, adjust `PATTERN` or `PHASE_DIR` above.

In [ ]:
image_paths = sorted(PHASE_DIR.glob(PATTERN))

print(f"Phase dir : {PHASE_DIR}")
print(f"Pattern   : {PATTERN}")
print(f"Found     : {len(image_paths)} image(s)")
print()

# preview first and last 5
show = image_paths[:5] + ([] if len(image_paths) <= 10 else ['...']) + image_paths[-5:]
for p in show:
    if p == '...':
        print('  ...')
    else:
        already_done = (OUTPUT_DIR / p.with_suffix('').with_suffix('').name / f"{p.with_suffix('').with_suffix('').name}.seg.npy").exists()
        status = "(already done)" if already_done and not IS_OVERWRITE else ""
        print(f"  {p.name}  {status}")

## Submit jobs

Submits one `bsub` job per image. Each job:
- Runs in the `omnipose` conda environment
- Uses `{LSF_MEMORY}` MB of RAM
- Writes stdout/stderr logs to `omnipose_seg/../lsf/`

If `IS_OVERWRITE = False`, FOVs that already have a `seg.npy` file are skipped (no job submitted).

In [ ]:
import subprocess

# log files go here — one .o and one .e per job
log_dir = OUTPUT_DIR.parent / "lsf"
log_dir.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

submitted = []
skipped   = []

for path in image_paths:
    base     = path.with_suffix('').with_suffix('').name   # fov_1_hyb_1.phase.tif → fov_1_hyb_1
    seg_path = OUTPUT_DIR / base / f"{base}.seg.npy"

    if not IS_OVERWRITE and seg_path.exists():
        skipped.append(base)
        continue

    job_name = f"{base}.omniseg"
    log_out  = log_dir / f"{job_name}.o"
    log_err  = log_dir / f"{job_name}.e"

    code = (
        f"conda run -n {CONDA_ENV} python {LSF_SCRIPT} "
        f"{path} {OUTPUT_DIR} {MODEL_TYPE} {IS_OVERWRITE}"
    )
    bsub_cmd = (
        f'bsub -J {job_name} -q {LSF_QUEUE} '
        f'-R "rusage[mem={LSF_MEMORY}]" '
        f'-o {log_out} -e {log_err} "{code}"'
    )

    result = subprocess.run(bsub_cmd, shell=True, capture_output=True, text=True)
    status = result.stdout.strip() or result.stderr.strip()
    print(f"  {base}: {status}")
    submitted.append(base)

print()
print(f"Submitted : {len(submitted)} job(s)")
print(f"Skipped   : {len(skipped)} (already done)")
print(f"Logs in   : {log_dir}")

## Monitor jobs

Run the cell below at any time to see which jobs are still running/pending.

| `bjobs` status | Meaning |
|---|---|
| `PEND` | Waiting in the queue |
| `RUN` | Currently running |
| `DONE` | Finished successfully |
| `EXIT` | Failed — check the `.e` log file |

When all jobs show `DONE`, you can open `omnipose_seg_qc.ipynb` on your laptop to review results.

In [ ]:
import subprocess

# show only jobs whose name ends with .omniseg
result = subprocess.run('bjobs -J "*.omniseg"', shell=True, capture_output=True, text=True)
print(result.stdout or "No omnipose jobs running (all done or not submitted yet).")
if result.stderr:
    print(result.stderr)

## Check which FOVs are done

Scans `OUTPUT_DIR` and reports which FOVs have a `seg.npy` file (finished) vs which are still missing.

In [ ]:
done    = []
missing = []

for path in image_paths:
    base     = path.with_suffix('').with_suffix('').name
    seg_path = OUTPUT_DIR / base / f"{base}.seg.npy"
    if seg_path.exists():
        done.append(base)
    else:
        missing.append(base)

print(f"Done    ({len(done)})   : {', '.join(done[:10])}{'...' if len(done) > 10 else ''}")
print(f"Missing ({len(missing)}): {', '.join(missing[:10])}{'...' if len(missing) > 10 else ''}")

## Inspect a log file

If a job failed (`EXIT` status), look at its `.e` log to see the error.  
Set `FOV_TO_CHECK` to the FOV name you want to inspect.

In [ ]:
FOV_TO_CHECK = "fov_1_hyb_1"   # ← change this to the FOV you want to inspect

log_dir = OUTPUT_DIR.parent / "lsf"
err_file = log_dir / f"{FOV_TO_CHECK}.omniseg.e"
out_file = log_dir / f"{FOV_TO_CHECK}.omniseg.o"

for label, f in [("STDOUT (.o)", out_file), ("STDERR (.e)", err_file)]:
    print(f"=== {label} ===")
    if f.exists():
        print(f.read_text(errors='replace')[-3000:])   # last 3000 chars
    else:
        print(f"  (file not found: {f})")
    print()